In [0]:
from pyspark.sql import functions as F, types as T

# ---- Catalog / schema constants ----
CATALOG = "opsanalytics_adb_workspace01"
LAB_SCHEMA = f"{CATALOG}.lab"                 # mapping tables live here
STAGING_SCHEMA = f"{CATALOG}.lab_staging"
PROD_SCHEMA = f"{CATALOG}.lab_production"

SCC_DIR = "/Volumes/opsanalytics_adb_workspace01/lab/raw_data/scc_data"

# ---- PK for the target table ----
SCC_PK = ["ORDERID", "TEST", "RESULTTIME", "TESTNAME"]

# ---- Load mapping tables from the lab schema ----
scc_test_code = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_test_codes")
scc_setting   = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_clinictype")
mshs_site     = spark.table(f"{LAB_SCHEMA}.lab_kpi_site_names")
scc_icu_raw   = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_icu")
tat_targets_raw = spark.table(f"{LAB_SCHEMA}.lab_kpi_turnaround_targets")

# ---- Derived columns on mapping tables (matching the R setup block) ----

# scc_icu: SiteCodeName = paste(SITE, WARD, WARD_NAME)  [R line 142]
scc_icu = scc_icu_raw.withColumn(
    "SiteCodeName",
    F.concat_ws(" ", F.col("SITE"), F.col("WARD"), F.col("WARD_NAME"))
)

# tat_targets: 3-tier Concate  [R lines 128-133]
#   PRIORITY == "All" & PT_SETTING == "All"  -> paste(TEST, DIVISION)
#   PRIORITY != "All" & PT_SETTING == "All"  -> paste(TEST, DIVISION, PRIORITY)
#   else                                     -> paste(TEST, DIVISION, PRIORITY, PT_SETTING)
tat_targets = tat_targets_raw.withColumn(
    "Concate",
    F.when(
        (F.col("PRIORITY") == "All") & (F.col("PT_SETTING") == "All"),
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"))
    ).when(
        (F.col("PRIORITY") != "All") & (F.col("PT_SETTING") == "All"),
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"), F.col("PRIORITY"))
    ).otherwise(
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"), F.col("PRIORITY"), F.col("PT_SETTING"))
    )
)

# ---- Constant lists / orderings (R lines 155-173) ----
CP_MICRO_LAB_ORDER = ["Troponin", "Lactate WB", "BUN", "HGB", "PT", "Rapid Flu", "C. diff"]
ALL_SITES = ["MSH", "MSQ", "MSB", "MSW", "MSM", "MSSN", "RTC"]
HOSP_SITES = ["MSH", "MSQ", "MSB", "MSW", "MSM", "MSSN"]
INFUSION_SITES = ["RTC"]

PT_SETTING_ORDER = ["ED", "ICU", "IP Non-ICU", "Amb", "Other"]
PT_SETTING_ORDER2 = ["ED & ICU", "IP Non-ICU", "Amb", "Other"]
DASHBOARD_PT_SETTING = ["ED & ICU", "IP Non-ICU", "Amb"]
DASHBOARD_PRIORITY_ORDER = ["All", "Stat", "Routine"]
CP_DIVISION_ORDER = ["Chemistry", "Hematology", "Microbiology RRL", "Infusion"]

In [0]:
def clean_scc_datetimes(df, cols):
    """Strip *...* junk and cast to timestamp."""
    for c in cols:
        # remove any *...* substring (failed-decode markers), then trim
        cleaned = F.regexp_replace(F.col(c).cast("string"), r"\*.*?\*", "")
        df = df.withColumn(c, F.to_timestamp(F.trim(cleaned), "yyyy-MM-dd HH:mm:ss"))
    return df

DATETIME_COLS = ["ORDERING_DATE", "COLLECTION_DATE", "RECEIVE_DATE", "VERIFIED_DATE"]

def preprocess_scc_4a(raw_scc):
    # --- Dedup entire rows (R line 812: unique) ---
    raw = raw_scc.dropDuplicates()

    # --- Strip *...* from ALL string columns (R lines 827-830) ---
    # R applies gsub across all character cols; do the same before datetime parse
    for c, dtype in raw.dtypes:
        if dtype == "string":
            raw = raw.withColumn(c, F.regexp_replace(F.col(c), r"\*.*?\*", ""))

    # --- Parse the 4 datetime columns (R lines 832-842) ---
    raw = clean_scc_datetimes(raw, DATETIME_COLS)

    # --- Join 1: test codes, then filter to in-scope (R lines 846-853) ---
    # TestIncl = !is.na(TEST) after join; keep only matched
    raw = raw.join(
        F.broadcast(scc_test_code),
        raw["TEST_ID"] == scc_test_code["SCC_TEST_ID"],
        how="left"
    ).drop("SCC_TEST_ID")
    raw = raw.filter(F.col("TEST").isNotNull())   # TestIncl filter

    # --- Join 2: clinic type / setting (R lines 856-857) ---
    raw = raw.join(F.broadcast(scc_setting), on="CLINIC_TYPE", how="left")

    # --- Join 3: site name via first char of COLLECT_CENTER_ID (R lines 859-863) ---
    raw = raw.withColumn("COLLECT_CENTER_ID", F.substring(F.col("COLLECT_CENTER_ID"), 1, 1))
    raw = raw.join(
        F.broadcast(mshs_site),
        raw["COLLECT_CENTER_ID"] == mshs_site["DATA_SITE"],
        how="left"
    ).drop("DATA_SITE")

    return raw

In [0]:
scc_pdf = 
scc_4a = preprocess_scc_4a(spark.createDataFrame(scc_pdf)) 
print(f"Rows after 4a (joined + in-scope filter): {scc_4a.count()}")
scc_4a.select("TEST_ID", "TEST", "CLINIC_TYPE", "COLLECT_CENTER_ID", "SITE").show(10, truncate=False)